In [33]:
import pandas as pd
import sqlite3
import json

### load the data set

In [34]:
orders=pd.read_csv(rf"C:\Users\yadla\Downloads\orders.csv")
orders.head()

,order_id,user_id,restaurant_id,order_date,total_amount,restaurant_name
0,1,2508,450,18-02-2023,842.97,New Foods Chinese
1,2,2693,309,18-01-2023,546.68,Ruchi Curry House Multicuisine
2,3,2084,107,15-07-2023,163.93,Spice Kitchen Punjabi
3,4,319,224,04-10-2023,1155.97,Darbar Kitchen Non-Veg
4,5,1064,293,25-12-2023,1321.91,Royal Eatery South Indian


In [23]:
users=pd.read_json(rf"C:\Users\yadla\Downloads\users.json")
users.head()

,user_id,name,city,membership
0,1,User_1,Chennai,Regular
1,2,User_2,Pune,Gold
2,3,User_3,Bangalore,Gold
3,4,User_4,Bangalore,Regular
4,5,User_5,Pune,Gold


In [54]:
sql_script = """
CREATE TABLE restaurants (
    restaurant_id INT,
    restaurant_name TEXT,
    cuisine TEXT,
    rating FLOAT
);

INSERT INTO restaurants VALUES (1, 'Restaurant_1', 'Chinese', 4.8);
INSERT INTO restaurants VALUES (2, 'Restaurant_2', 'Indian', 4.1);
INSERT INTO restaurants VALUES (3, 'Restaurant_3', 'Mexican', 4.3);
INSERT INTO restaurants VALUES (4, 'Restaurant_4', 'Chinese', 4.1);
"""

In [55]:
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()
cursor.executescript(sql_script)

In [56]:
restaurants_df = pd.read_sql_query("SELECT * FROM restaurants", conn)

In [57]:
print("Success! Restaurants data is now in a DataFrame.")
display(restaurants_df.head())

Success! Restaurants data is now in a DataFrame.


,restaurant_id,restaurant_name,cuisine,rating
0,1,Restaurant_1,Chinese,4.8
1,2,Restaurant_2,Indian,4.1
2,3,Restaurant_3,Mexican,4.3
3,4,Restaurant_4,Chinese,4.1


### merging the det set

In [58]:
merged_df = pd.merge(orders, users, on='user_id', how='left')

In [65]:
final_df = pd.merge(merged_df, restaurants_df, on='restaurant_id', how='left', suffixes=('', '_sql'))

In [66]:
final_df.head()

,order_id,user_id,restaurant_id,order_date,total_amount,restaurant_name,name,city,membership,restaurant_name_sql,cuisine,rating
0,1,2508,450,18-02-2023,842.97,New Foods Chinese,User_2508,Hyderabad,Regular,NaN,NaN,NaN
1,2,2693,309,18-01-2023,546.68,Ruchi Curry House Multicuisine,User_2693,Pune,Regular,NaN,NaN,NaN
2,3,2084,107,15-07-2023,163.93,Spice Kitchen Punjabi,User_2084,Chennai,Gold,NaN,NaN,NaN
3,4,319,224,04-10-2023,1155.97,Darbar Kitchen Non-Veg,User_319,Bangalore,Gold,NaN,NaN,NaN
4,5,1064,293,25-12-2023,1321.91,Royal Eatery South Indian,User_1064,Pune,Regular,NaN,NaN,NaN


In [67]:
final_df.to_csv('final_food_delivery_dataset.csv', index=False)
print("Data Merge Complete!")

Data Merge Complete!


### 1. Total Orders by Gold Members

In [71]:
gold_orders = final_df[final_df['membership'] == 'Gold'].shape[0]
gold_orders

4987

In [72]:
hyd_rev = round(final_df[final_df['city'] == 'Hyderabad']['total_amount'].sum())
hyd_rev

1889367

In [74]:
distinct_users = final_df['user_id'].nunique()
distinct_users

2883

In [75]:
gold_aov = round(final_df[final_df['membership'] == 'Gold']['total_amount'].mean(), 2)
gold_aov

797.15

In [78]:
high_rating_orders = final_df[final_df['rating'] >= 4.5].shape[0]
high_rating_orders

16

In [80]:
gold_data = final_df[final_df['membership'] == 'Gold']
top_gold_city = gold_data.groupby('city')['total_amount'].sum().idxmax()
top_city_orders = gold_data[gold_data['city'] == top_gold_city].shape[0]
print(gold_data)
print(top_gold_city)
print(top_city_orders)

      order_id  user_id  restaurant_id  order_date  total_amount  \
2            3     2084            107  15-07-2023        163.93   
3            4      319            224  04-10-2023       1155.97   
8            9      364              7  05-12-2023        953.30   
11          12      884            423  27-10-2023       1484.65   
13          14      364            112  24-09-2023        898.24   
...        ...      ...            ...         ...           ...   
9993      9994     1616            198  28-04-2023        322.54   
9994      9995     1257            328  12-11-2023        137.96   
9995      9996     2528            249  21-05-2023       1211.96   
9997      9998      522            420  11-11-2023        979.44   
9998      9999      319            492  08-09-2023       1105.93   

                     restaurant_name       name       city membership  \
2              Spice Kitchen Punjabi  User_2084    Chennai       Gold   
3             Darbar Kitchen Non-Veg 

In [81]:
# Print Results
print(f"Total Gold Orders: {gold_orders}")
print(f"Hyderabad Total Revenue: {hyd_rev}")
print(f"Distinct Users: {distinct_users}")
print(f"Gold Average Order Value: {gold_aov}")
print(f"Orders (Rating >= 4.5): {high_rating_orders}")
print(f"Orders in Top Gold City ({top_gold_city}): {top_city_orders}")

Total Gold Orders: 4987
Hyderabad Total Revenue: 1889367
Distinct Users: 2883
Gold Average Order Value: 797.15
Orders (Rating >= 4.5): 16
Orders in Top Gold City (Chennai): 1337


In [82]:
gold_data = final_df[final_df['membership'] == 'Gold']
top_gold_city = gold_data.groupby('city')['total_amount'].sum().idxmax()
print(f"1. Highest revenue city (Gold): {top_gold_city}")

1. Highest revenue city (Gold): Chennai


In [83]:
top_cuisine_aov = final_df.groupby('cuisine')['total_amount'].mean().idxmax()
print(f"2. Cuisine with highest AOV: {top_cuisine_aov}")

2. Cuisine with highest AOV: Mexican


In [84]:
user_totals = final_df.groupby('user_id')['total_amount'].sum()
users_above_1000 = user_totals[user_totals > 1000].count()
print(f"3. Distinct users with total > ₹1000: {users_above_1000}")

3. Distinct users with total > ₹1000: 2544


In [85]:
bins = [3.0, 3.5, 4.0, 4.5, 5.0]
labels = ['3.0-3.5', '3.6-4.0', '4.1-4.5', '4.6-5.0']
final_df['rating_range'] = pd.cut(final_df['rating'], bins=bins, labels=labels)
top_rating_range = final_df.groupby('rating_range')['total_amount'].sum().idxmax()
print(f"4. Top revenue rating range: {top_rating_range}")

4. Top revenue rating range: 4.1-4.5


C:\Users\yadla\AppData\Local\Temp\ipykernel_25472\63254211.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  top_rating_range = final_df.groupby('rating_range')['total_amount'].sum().idxmax()


In [86]:
top_gold_aov_city = gold_data.groupby('city')['total_amount'].mean().idxmax()
print(f"5. Highest AOV city (Gold): {top_gold_aov_city}")

5. Highest AOV city (Gold): Chennai


In [87]:
cuisine_stats = final_df.groupby('cuisine').agg({'restaurant_id': 'nunique', 'total_amount': 'sum'})
print(f"6. Cuisine analysis:\n{cuisine_stats}")

6. Cuisine analysis:
         restaurant_id  total_amount
cuisine                             
Chinese              2      23378.53
Indian               1      14081.16
Mexican              1      14990.90


In [88]:
gold_pct = round((len(gold_data) / len(final_df)) * 100)
print(f"7. Gold orders percentage: {gold_pct}%")

7. Gold orders percentage: 50%


In [89]:
rest_stats = final_df.groupby('restaurant_name').agg({'total_amount': ['mean', 'count']})
rest_stats.columns = ['aov', 'order_count']
filtered_rests = rest_stats[rest_stats['order_count'] < 20]
top_low_vol_rest = filtered_rests['aov'].idxmax()
print(f"8. Highest AOV rest (< 20 orders): {top_low_vol_rest}")

8. Highest AOV rest (< 20 orders): Hotel Dhaba Multicuisine


In [90]:
combo_revenue = final_df.groupby(['membership', 'cuisine'])['total_amount'].sum().idxmax()
print(f"9. Top revenue combination: {combo_revenue}")

9. Top revenue combination: ('Regular', 'Chinese')


In [91]:
final_df['order_date'] = pd.to_datetime(final_df['order_date'], dayfirst=True)
final_df['quarter'] = final_df['order_date'].dt.quarter
top_quarter = final_df.groupby('quarter')['total_amount'].sum().idxmax()
print(f"10. Highest revenue Quarter: Q{top_quarter}")

10. Highest revenue Quarter: Q3
